# RAG on Azure — Day 12: Contextual Retrieval 

**The problem:** chunking destroys context. A chunk that reads
*"Revenue grew 3.4% over the previous quarter"* is perfectly clear inside its
document — and perfectly **ambiguous** once it's a standalone record in a vector
index. Which company? Which quarter? The embedding has no idea, so retrieval
guesses.

**Today's fix (contextual retrieval):** at **indexing time**, ask the LLM to write
a 1–2 sentence note situating each chunk within its source document
(*"This chunk is from NorthPeak Logistics' Q2 FY2026 shareholder report and
discusses quarterly revenue growth."*) and **prepend it to the chunk before
embedding**. The index gets smarter; queries and prompts stay unchanged.

Contrast with Days 10–11: parent-document and sentence-window **expand context at
query time**. Contextual retrieval **enriches context at indexing time**. They are
complementary, not competing.

**Pipeline today:**

1. Load three quarterly-report PDFs (two companies; overlapping quarters — built to be ambiguous).
2. Chunk by section, build a **baseline index** from raw chunks.
3. For each chunk, generate a *situating context* with one LLM call, prepend it, build a **contextual index**.
4. Run the same ambiguous queries against both indexes and compare what comes back.

## 1. Setup & configuration

Same config pattern as Days 1–11. If you've been following along, the helpers
live in `common/`; the `except ImportError` block makes this notebook
self-contained if you're running it standalone.

In [ ]:
import os, json, re, time, hashlib

try:
    # Shared utilities built up over Days 1-11
    from common.azure_clients import load_config, get_openai_client, get_search_client
except ImportError:
    from openai import AzureOpenAI
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    def load_config(path="config.json"):
        """Load config.json if present, else fall back to environment variables."""
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        return {
            "openai_endpoint":     os.environ["AZURE_OPENAI_ENDPOINT"],
            "openai_key":          os.environ["AZURE_OPENAI_KEY"],
            "search_endpoint":     os.environ["AZURE_SEARCH_ENDPOINT"],
            "search_key":          os.environ["AZURE_SEARCH_KEY"],
            "chat_deployment":     os.environ.get("AZURE_CHAT_DEPLOYMENT", "gpt-4o-mini"),
            "embedding_deployment": os.environ.get("AZURE_EMBED_DEPLOYMENT", "text-embedding-3-small"),
        }

    def get_openai_client(config):
        return AzureOpenAI(
            azure_endpoint=config["openai_endpoint"],
            api_key=config["openai_key"],
            api_version="2024-10-21",
        )

    def get_search_client(config, index_name):
        return SearchClient(
            endpoint=config["search_endpoint"],
            index_name=index_name,
            credential=AzureKeyCredential(config["search_key"]),
        )

config  = load_config()
openai  = get_openai_client(config)

INDEX_RAW = "rag-finance-day12-raw"   # baseline: chunks as-is
INDEX_CTX = "rag-finance-day12-ctx"   # contextual: LLM-situated chunks
DATA_DIR  = "data-day12"

print("Config loaded. Chat:", config["chat_deployment"],
      "| Embeddings:", config["embedding_deployment"])

## 2. Load & chunk the corpus

The corpus is **designed to break naive RAG**: two companies (NorthPeak
Logistics, BlueHarbor Retail), overlapping quarters, near-identical section
structure and phrasing. Section text rarely names the company or quarter —
that information lives only in the report header.

We chunk **one section per chunk** (the natural unit for these reports). Each
chunk keeps a reference to its full parent document, which we'll need in step 4.

In [ ]:
from pypdf import PdfReader

def load_pdf_text(path):
    return "\n".join(page.extract_text() for page in PdfReader(path).pages)

SECTION_HEADINGS = [
    "Financial Highlights", "Revenue Performance",
    "Margins and Costs", "Outlook", "Principal Risks",
]

def split_into_sections(full_text):
    """Split a report into (heading, body) pairs using the known headings."""
    pattern = "(" + "|".join(re.escape(h) for h in SECTION_HEADINGS) + ")"
    parts = re.split(pattern, full_text)
    header = parts[0].strip()                      # title + subtitle block
    sections = []
    for i in range(1, len(parts) - 1, 2):
        sections.append((parts[i].strip(), parts[i + 1].strip()))
    return header, sections

documents, chunks = {}, []
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith(".pdf"):
        continue
    full_text = load_pdf_text(os.path.join(DATA_DIR, fname))
    documents[fname] = full_text
    header, sections = split_into_sections(full_text)
    for heading, body in sections:
        chunks.append({
            "id": hashlib.md5(f"{fname}-{heading}".encode()).hexdigest()[:12],
            "source": fname,
            "section": heading,
            "content": body,
        })

print(f"Loaded {len(documents)} documents -> {len(chunks)} section chunks\n")
print("Example of the ambiguity problem — this chunk alone:\n")
example = next(c for c in chunks if c["section"] == "Financial Highlights")
print(f'  [{example["source"]} / {example["section"]}]')
print(" ", example["content"][:200], "...")
print("\nNothing in the body says WHICH company or WHICH quarter. That's the bug.")

## 3. Baseline index — raw chunks

Standard Day 2-style pipeline: embed each chunk's text and upload. This is our
control group.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    SearchFieldDataType, VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)
from azure.core.credentials import AzureKeyCredential

EMBED_DIMS = 1536  # text-embedding-3-small

def embed(texts, batch=16):
    """Embed a list of strings with the Azure OpenAI embeddings deployment."""
    out = []
    for i in range(0, len(texts), batch):
        resp = openai.embeddings.create(
            model=config["embedding_deployment"], input=texts[i:i+batch])
        out.extend(d.embedding for d in resp.data)
    return out

def build_index(index_name):
    """(Re)create a minimal vector index: id, source, section, content, vector."""
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="section", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SearchField(name="contentVector",
                    type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                    searchable=True, vector_search_dimensions=EMBED_DIMS,
                    vector_search_profile_name="vp"),
    ]
    vs = VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
        profiles=[VectorSearchProfile(name="vp", algorithm_configuration_name="hnsw")],
    )
    ic = SearchIndexClient(config["search_endpoint"], AzureKeyCredential(config["search_key"]))
    try:
        ic.delete_index(index_name)
    except Exception:
        pass
    ic.create_index(SearchIndex(name=index_name, fields=fields, vector_search=vs))
    print(f"Index '{index_name}' created.")

def upload(index_name, records, vectors):
    client = get_search_client(config, index_name)
    docs = [{**r, "contentVector": v} for r, v in zip(records, vectors)]
    client.upload_documents(documents=docs)
    time.sleep(2)  # let the index refresh
    print(f"Uploaded {len(docs)} records to '{index_name}'.")

# --- build + populate the RAW baseline ---
build_index(INDEX_RAW)
raw_vectors = embed([c["content"] for c in chunks])
upload(INDEX_RAW, chunks, raw_vectors)

## 4. The Day 12 step — generate situating context per chunk

For every chunk we make **one LLM call** that sees (a) the *whole document* and
(b) the chunk, and returns 1–2 sentences locating the chunk in its document —
company, report, period, topic. We prepend that note to the chunk text and
**embed the combined string**.

Two production notes:

- **Cost.** This is `O(corpus size × LLM calls)` at indexing time — but it's a
  *one-time* cost per document, and **prompt caching** makes it cheap: the
  (large, identical) document prefix is cached across the chunk calls, so you
  pay full price for it only once per document.
- **Determinism.** `temperature=0` — we want stable, factual context notes,
  not creative writing.

In [ ]:
CONTEXT_PROMPT = """<document>
{document}
</document>

Here is a chunk taken from the document above:

<chunk>
{chunk}
</chunk>

Write 1-2 short sentences situating this chunk within the overall document so
the chunk can be understood on its own. Always name the organization and the
reporting period. Respond with ONLY the situating sentences, nothing else."""

def situate(document_text, chunk_text):
    resp = openai.chat.completions.create(
        model=config["chat_deployment"],
        temperature=0,
        max_tokens=120,
        messages=[{
            "role": "user",
            "content": CONTEXT_PROMPT.format(document=document_text, chunk=chunk_text),
        }],
    )
    return resp.choices[0].message.content.strip()

ctx_chunks = []
for c in chunks:
    note = situate(documents[c["source"]], c["content"])
    ctx_chunks.append({**c, "content": f"{note}\n\n{c['content']}"})
    print(f"[{c['source']} / {c['section']}]")
    print(f"   context: {note}\n")

Notice what just happened: the chunk
*"Revenue ... an increase of 3.4% over the previous quarter"* now carries a
prefix like *"This chunk is from NorthPeak Logistics Inc.'s Q2 FY2026
shareholder report..."* — and that prefix becomes part of the **embedding**.
The company name and quarter are now in vector space.

In [ ]:
# --- build + populate the CONTEXTUAL index ---
build_index(INDEX_CTX)
ctx_vectors = embed([c["content"] for c in ctx_chunks])
upload(INDEX_CTX, ctx_chunks, ctx_vectors)

## 5. Head-to-head — same queries, both indexes

The acid test: queries whose answer depends on knowing **which document** a
chunk came from. With three near-identical reports in the index, raw chunks
should confuse companies and quarters; contextual chunks should not.

In [ ]:
from azure.search.documents.models import VectorizedQuery

def retrieve(index_name, question, k=3):
    qv = embed([question])[0]
    client = get_search_client(config, index_name)
    results = client.search(
        search_text=None,
        vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=k,
                                        fields="contentVector")],
        select=["source", "section", "content"], top=k,
    )
    return list(results)

def answer(question, hits):
    context = "\n\n---\n\n".join(
        f"[{h['source']} / {h['section']}]\n{h['content']}" for h in hits)
    resp = openai.chat.completions.create(
        model=config["chat_deployment"],
        temperature=0,
        messages=[
            {"role": "system",
             "content": "Answer ONLY from the provided context. "
                        "If the context is insufficient or ambiguous, say so. "
                        "Cite the source file you used."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return resp.choices[0].message.content.strip()

def compare(question, k=3):
    print("=" * 88)
    print(f"Q: {question}\n")
    for label, index in [("RAW CHUNKS (baseline)", INDEX_RAW),
                         ("CONTEXTUAL CHUNKS (Day 12)", INDEX_CTX)]:
        hits = retrieve(index, question, k)
        print(f"-- {label} --")
        for h in hits:
            print(f"   retrieved: {h['source']} / {h['section']}")
        print(f"   Answer: {answer(question, hits)}\n")

### Demo A — "which company?" ambiguity

Both Q2 reports report 3.4% growth. Raw embeddings can't tell the chunks apart
because neither chunk body names its company. The contextual index can.

In [ ]:
compare("How much did NorthPeak's revenue grow in Q2 2026, and what drove it?")

### Demo B — "which quarter?" ambiguity

NorthPeak's Q1 and Q2 margin sections are structural twins ("Gross margin
compressed/recovered by N basis points, driven by..."). Only document context
distinguishes them.

In [ ]:
compare("Why did NorthPeak's gross margin improve in the second quarter?")

### Demo C — cross-company comparison

This needs the right chunk from **each** company. Watch whether the baseline
pulls two chunks from the same report.

In [ ]:
compare("Compare the principal risks NorthPeak and BlueHarbor flagged in Q2 2026.", k=4)

## 6. A tiny retrieval scorecard

Eight context-dependent questions with known correct sources. We count a hit
when the expected document appears in the top-3. (Day 14 turns this idea into a
proper harness.)

In [ ]:
EVAL = [
    ("What is NorthPeak's revenue guidance for Q3 2026?",          "northpeak_q2_2026.pdf"),
    ("What drove NorthPeak's growth in Q1 2026?",                  "northpeak_q1_2026.pdf"),
    ("What dividend did NorthPeak's board approve?",               "northpeak_q2_2026.pdf"),
    ("Why did BlueHarbor's gross margin decline?",                 "blueharbor_q2_2026.pdf"),
    ("What share of BlueHarbor's revenue is e-commerce?",          "blueharbor_q2_2026.pdf"),
    ("What fleet maintenance issue affected NorthPeak's margins?", "northpeak_q1_2026.pdf"),
    ("What is BlueHarbor's biggest risk for next quarter?",        "blueharbor_q2_2026.pdf"),
    ("How many drivers does NorthPeak need to hire?",              "northpeak_q2_2026.pdf"),
]

def score(index_name):
    hits = 0
    for q, expected in EVAL:
        sources = [h["source"] for h in retrieve(index_name, q, k=3)]
        ok = expected in sources
        hits += ok
        print(f"  {'PASS' if ok else 'MISS'}  {q}")
    return hits

print("RAW baseline:")
raw_score = score(INDEX_RAW)
print(f"\nCONTEXTUAL:")
ctx_score = score(INDEX_CTX)
print(f"\nTop-3 source accuracy — raw: {raw_score}/{len(EVAL)}  |  contextual: {ctx_score}/{len(EVAL)}")

## 7. Takeaways

- **Chunking is lossy by construction.** Every chunk silently depends on
  document-level facts (who, when, what report) that embedding raw text throws
  away. Contextual retrieval puts those facts back *before* embedding.
- **Indexing-time vs query-time context.** Days 10–11 expanded context after
  retrieval; today made retrieval itself smarter. In production the techniques
  stack: contextual chunks + reranking (Day 9) is a particularly strong combo.
- **Cost is front-loaded and cacheable.** One LLM call per chunk, once, at
  ingestion — with prompt caching the per-document text is paid for roughly once.
  Compare that to query-time techniques, which cost on every single query.
- **Hybrid search amplifies the win.** The situating note injects exact terms
  ("NorthPeak", "Q2 FY2026") into the chunk, so the BM25 half of hybrid search
  (Day 8) benefits too, not just the vector half.

### Next up — Day 13: metadata-aware retrieval at scale
Contextual notes put document identity into the *embedding*. Day 13 puts it
into the *schema* — structured fields (`date`, `author`, `docType`, `version`)
you can filter and facet on, which is how you slice corpora with thousands of
documents.